# 11b — PySpark DataFrames and windows — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/spark.txt and a compatible Java runtime. This starts local Spark.

[Stage guide](../stages/11.md) · [Course map](../PLAN.md) · [Project](../../projects/stage11/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Create a real Spark session
- Aggregate and window typed records
- Read and write Parquet with validation


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

This notebook requires the Spark environment and a compatible Java runtime. It runs a genuine local Spark job. Local mode uses one machine; a distributed cluster adds scheduling, remote storage, executor failures, and resource configuration.

Prefer DataFrame expressions to Python row UDFs when an equivalent built-in exists: the optimizer can inspect expressions and avoid Python serialization. Use explicit types, push filters early, and select needed columns. Call explain to inspect scans, exchanges, and joins. Caching only helps reused expensive intermediates and must fit memory.

Window ordering needs a tie-breaker. A ROWS frame counts records; a RANGE frame follows ordering values. Three rows are not necessarily three calendar days. A production pipeline also needs ingestion IDs, deduplication, late-data policy, and idempotent output. Stop the session after the experiment to release resources.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable  # Local workers must use this notebook's Python.
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.master("local[2]").appName("ai-learning").config("spark.ui.enabled", "false").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.createDataFrame([(1, "A", 10), (2, "A", 20), (3, "B", 5)], "id long, account string, cents long")
df.show()


## Exercise 1: Group totals

Return account,sum_cents sorted by account, using Spark DataFrame expressions.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def totals(df):
    return df.groupBy("account").agg(F.sum("cents").alias("sum_cents")).orderBy("account")


In [ ]:
assert [(r.account, r.sum_cents) for r in totals(df).collect()] == [("A", 30), ("B", 5)]
print("Exercise 1: checks passed")


**Why this works:** Production jobs should write large outputs instead of collecting to the driver.


## Exercise 2: Running total

Add running_cents partitioned by account ordered by unique id with an unbounded preceding ROWS frame.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def with_running(df):
    window = Window.partitionBy("account").orderBy("id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    return df.withColumn("running_cents", F.sum("cents").over(window))


In [ ]:
assert [r.running_cents for r in with_running(df).orderBy("id").collect()] == [10, 30, 5]
print("Exercise 2: checks passed")


**Why this works:** Stable ordering is necessary for reproducible cumulative values.


## Exercise 3: Parquet round trip

Write df to a supplied new directory using errorifexists, read it, and return row count. Do not overwrite existing work.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def parquet_roundtrip(df, path):
    df.write.mode("errorifexists").parquet(str(path))
    return spark.read.parquet(str(path)).count()


In [ ]:
import tempfile
from pathlib import Path
with tempfile.TemporaryDirectory() as folder:
    assert parquet_roundtrip(df, Path(folder)/"data") == 3
spark.stop()
print("Exercise 3: checks passed")


**Why this works:** A round trip must validate schema and aggregates too before being trusted with real data.


## Independent transfer

Run the Stage 11 job, compare partitions 2/8/32, capture explain plans and timings, and compute anomalies plus rolling statistics. Repeat on an actual cluster before claiming distributed operational experience.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why lazy execution?

   It lets Spark optimize a whole plan before running actions.

2. What does a shuffle cost?

   Network transfer, serialization, sorting, disk spill, and coordination.


## Reading and review

- [Primary reference 1](https://spark.apache.org/docs/latest/api/python/getting_started/install.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
